In [2]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 8.6 MB/s eta 0:00:00


In [3]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 99.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found 

In [6]:
import os
import json
from groq import Groq


MODEL = "openai/gpt-oss-20b"


def get_client():
    api_key =groq_api_key

    if not api_key:
        raise RuntimeError(
            "GROQ_API_KEY is not set. "
            "Set it as an environment variable before running the program."
        )

    return Groq(api_key=api_key)


def analyze_resume(resume_text: str, job_description: str) -> dict:
    client = get_client()

    system_prompt = """
You are an AI Resume & Job Match Assistant.

Your job is to help a job seeker understand how their resume
relates to a specific job description.

IMPORTANT RULES:

1. Analyze ONLY the resume and job description provided.
2. Never invent skills, jobs, projects, certifications, employers,
   achievements, dates, or years of experience.
3. A skill can be listed as matched only when the resume provides
   evidence for it.
4. If a job requirement is not clearly supported by the resume,
   place it in missing_or_unclear_skills.
5. Do not make hiring decisions.
6. Do not claim that the candidate will get or lose the job.
7. Give practical resume-improvement suggestions.
8. Keep the suggestions realistic and based on the supplied text.
9. Generate interview topics from the job requirements.
10. Return ONLY the requested structured JSON output.
"""

    user_prompt = f"""
CANDIDATE RESUME
----------------
{resume_text}

JOB DESCRIPTION
---------------
{job_description}

Analyze the resume against the job description.
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "resume_job_analysis",
                "strict": True,
                "schema": {
                    "type": "object",
                    "properties": {

                        "summary": {
                            "type": "string"
                        },

                        "matched_skills": {
                            "type": "array",
                            "items": {
                                "type": "string"
                            }
                        },

                        "missing_or_unclear_skills": {
                            "type": "array",
                            "items": {
                                "type": "string"
                            }
                        },

                        "resume_improvements": {
                            "type": "array",
                            "items": {
                                "type": "string"
                            }
                        },

                        "interview_topics": {
                            "type": "array",
                            "items": {
                                "type": "string"
                            }
                        },

                        "evidence_notes": {
                            "type": "array",
                            "items": {
                                "type": "string"
                            }
                        },

                        "disclaimer": {
                            "type": "string"
                        }
                    },

                    "required": [
                        "summary",
                        "matched_skills",
                        "missing_or_unclear_skills",
                        "resume_improvements",
                        "interview_topics",
                        "evidence_notes",
                        "disclaimer"
                    ],

                    "additionalProperties": False
                }
            }
        }
    )

    content = response.choices[0].message.content

    if not content:
        raise RuntimeError("The model returned an empty response.")

    return json.loads(content)


def print_report(result: dict) -> None:

    print("\n" + "=" * 70)
    print("AI RESUME & JOB MATCH ASSISTANT")
    print("=" * 70)

    print("\nSUMMARY")
    print(result["summary"])

    print("\nMATCHED SKILLS")
    for skill in result["matched_skills"]:
        print(f"- {skill}")

    print("\nMISSING OR UNCLEAR SKILLS")
    for skill in result["missing_or_unclear_skills"]:
        print(f"- {skill}")

    print("\nRESUME IMPROVEMENTS")
    for suggestion in result["resume_improvements"]:
        print(f"- {suggestion}")

    print("\nINTERVIEW TOPICS")
    for topic in result["interview_topics"]:
        print(f"- {topic}")

    print("\nEVIDENCE NOTES")
    for note in result["evidence_notes"]:
        print(f"- {note}")

    print("\nDISCLAIMER")
    print(result["disclaimer"])


def main():

    print("AI Resume & Job Match Assistant")
    print("--------------------------------")

    print("\nPaste your resume.")
    print("When finished, type END on a new line.\n")

    resume_lines = []

    while True:
        line = input()

        if line.strip() == "END":
            break

        resume_lines.append(line)

    resume = "\n".join(resume_lines).strip()

    if not resume:
        print("Resume cannot be empty.")
        return

    print("\nPaste the job description.")
    print("When finished, type END on a new line.\n")

    job_lines = []

    while True:
        line = input()

        if line.strip() == "END":
            break

        job_lines.append(line)

    job_description = "\n".join(job_lines).strip()

    if not job_description:
        print("Job description cannot be empty.")
        return

    try:
        result = analyze_resume(
            resume,
            job_description
        )

        print_report(result)

    except Exception as error:
        print("\nERROR:")
        print(error)


if __name__ == "__main__":
    main()

AI Resume & Job Match Assistant
--------------------------------

Paste your resume.
When finished, type END on a new line.

Hamid Nazir  Computer Science Student  Skills: Python C++ HTML CSS JavaScript Git Basic Machine Learning  Projects: - Built a Python student management system. - Created a machine learning project using KNN. - Built a simple web application using HTML, CSS and JavaScript.  Education: BS Computer Science
END

Paste the job description.
When finished, type END on a new line.

Junior Python Developer  Requirements: - Python programming - Git and GitHub - REST APIs - SQL databases - Problem-solving skills - Basic knowledge of machine learning - Ability to work with JavaScript is a plus
END

AI RESUME & JOB MATCH ASSISTANT

SUMMARY
The resume demonstrates core Python, Git, JavaScript, and basic machine learning experience, but lacks explicit evidence of REST API design, SQL database usage, and problem-solving examples required for the Junior Python Developer role.

MA